In [15]:
%pip install pypdf


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import os
import re
import glob
import pandas as pd
from pypdf import PdfReader

FACTSHEET_DIR = "data/factsheets"
PARSED_DIR = "data/parsed"

os.makedirs(PARSED_DIR, exist_ok=True)


def get_latest_pdf():
    pdf_files = glob.glob(os.path.join(FACTSHEET_DIR, "*.pdf"))
    if not pdf_files:
        raise FileNotFoundError("data/factsheets 폴더에 PDF가 없습니다.")
    pdf_files.sort()
    return pdf_files[-1]


def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    texts = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            texts.append(text)
    return "\n".join(texts)


def extract_asof_date(text):
    patterns = [
        r"Data as at:\s*([0-9]{1,2}\s+[A-Za-z]+\s+[0-9]{4})",
        r"Source: FTSE Russell as at\s*([0-9]{1,2}\s+[A-Za-z]+\s+[0-9]{4})"
    ]
    for p in patterns:
        m = re.search(p, text)
        if m:
            return m.group(1)
    return "N/A"


def extract_ytd_return(text):
    # Performance and Volatility 표에서
    # FTSE EPRA Nareit Developed 행의 3M, 6M, YTD, 12M 중 YTD 추출
    m = re.search(
        r"FTSE EPRA Nareit Developed\s+([\-0-9.]+)\s+([\-0-9.]+)\s+([\-0-9.]+)\s+([\-0-9.]+)",
        text
    )
    if m:
        return float(m.group(3))   # 세 번째 숫자가 YTD
    return None


def extract_number(pattern, text, as_int=False):
    m = re.search(pattern, text, re.S)
    if not m:
        return None

    value = m.group(1).replace(",", "").strip()
    try:
        return int(float(value)) if as_int else float(value)
    except:
        return None


def extract_index_characteristics(text):
    return {
        "number_of_constituents": extract_number(
            r"Number of constituents\s+([\d,]+)", text, as_int=True
        ),
        "net_mcap_usdm": extract_number(
            r"Net MCap (USDm)\s+([\d,]+)", text, as_int=True
        ),
        "dividend_yield_pct": extract_number(
            r"Dividend Yield %\s+([\d.]+)", text
        ),
        "largest_weight_pct": extract_number(
            r"Weight of Largest Constituent $$$%$$$\s+([\d.]+)", text
        ),
        "top10_weight_pct": extract_number(
            r"Top 10 Holdings $$$% Index MCap$$$\s+([\d.]+)", text
        ),
    }


def extract_country_breakdown(text):
    rows = []

    block_match = re.search(
        r"Country/Market Breakdown(.*?)(?:INFORMATION|Index Characteristics|3\s+of\s+3|Totals\s+354\s+[\d,]+\s+100\.00)",
        text,
        re.S
    )

    if not block_match:
        return pd.DataFrame(columns=["country", "no_of_cons", "net_mcap_usdm", "weight"])

    block = block_match.group(1)

    pattern = re.compile(r"([A-Za-z/&\-\s]+?)\s+(\d+)\s+([\d,]+)\s+([\d.]+)")
    matches = pattern.findall(block)

    for country, n_cons, mcap, weight in matches:
        country = " ".join(country.split()).strip()

        if country.lower() in ["country", "market", "country/market", "totals"]:
            continue

        rows.append({
            "country": country,
            "no_of_cons": int(n_cons),
            "net_mcap_usdm": int(mcap.replace(",", "")),
            "weight": float(weight)
        })

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values("weight", ascending=False).reset_index(drop=True)

    return df


def build_overview_csv(asof_date, ytd_return, characteristics):
    overview = pd.DataFrame([
        {"metric": "As of Date", "value": asof_date, "unit": ""},
        {"metric": "YTD Return", "value": ytd_return, "unit": "%"},
        {"metric": "Dividend Yield", "value": characteristics.get("dividend_yield_pct"), "unit": "%"},
        {"metric": "Number of Constituents", "value": characteristics.get("number_of_constituents"), "unit": ""},
        {"metric": "Net Market Cap", "value": characteristics.get("net_mcap_usdm"), "unit": "USDm"},
        {"metric": "Top 10 Weight", "value": characteristics.get("top10_weight_pct"), "unit": "%"},
        {"metric": "Largest Constituent Weight", "value": characteristics.get("largest_weight_pct"), "unit": "%"},
    ])
    overview.to_csv(os.path.join(PARSED_DIR, "overview_latest.csv"), index=False)


def main():
    pdf_path = get_latest_pdf()
    print(f"Using PDF: {pdf_path}")

    text = extract_text_from_pdf(pdf_path)

    asof_date = extract_asof_date(text)
    ytd_return = extract_ytd_return(text)
    characteristics = extract_index_characteristics(text)
    country_df = extract_country_breakdown(text)

    build_overview_csv(asof_date, ytd_return, characteristics)
    country_df.to_csv(os.path.join(PARSED_DIR, "country_weight_latest.csv"), index=False)

    print("Saved:")
    print("- data/parsed/overview_latest.csv")
    print("- data/parsed/country_weight_latest.csv")
    print("\nOverview preview:")
    print(pd.read_csv(os.path.join(PARSED_DIR, "overview_latest.csv")))
    print("\nCountry preview:")
    print(country_df.head())


if __name__ == "__main__":
    main()


Using PDF: data/factsheets\ENGL_20260430.pdf
Saved:
- data/parsed/overview_latest.csv
- data/parsed/country_weight_latest.csv

Overview preview:
                       metric          value  unit
0                  As of Date  30 April 2026   NaN
1                  YTD Return           10.0     %
2              Dividend Yield           3.76     %
3      Number of Constituents            354   NaN
4              Net Market Cap            NaN  USDm
5               Top 10 Weight            NaN     %
6  Largest Constituent Weight            NaN     %

Country preview:
     country  no_of_cons  net_mcap_usdm  weight
0        USA          97        1295886   64.03
1      Japan          58         183739    9.08
2  Australia          28         118171    5.84
3  Hong Kong          14          69442    3.43
4         UK          28          63802    3.15
